# Understanding Components of Transformer Architecture

## Imports

In [2]:
import os
import urllib
import torch
import torch.nn as nn
from torch.nn import functional as F

## Downloading the Dataset

In [3]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
path = r"C:\Users\sarve\Documents\Personal Experiments\GPT From Scratch\input.txt"

if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path}")
else:
    print(f"Already exists: {path}")

Already exists: C:\Users\sarve\Documents\Personal Experiments\GPT From Scratch\input.txt


## Inspecting the dataset

In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    print(f"length of dataset in characters: {len(text)}")
    print(text[:500])

length of dataset in characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [5]:
## Inspect all the unique characters that occur in this text

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Unique characters in the text: {''.join(chars)}")
print(f"Vocab Size: {vocab_size}")

Unique characters in the text: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab Size: 65


## Encoding and Decoding

In [6]:
## create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] ## encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) ## decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
## Encoding the entire text dataset and store it in torch.Tensor

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:500]) # the 500 characters we looked at earlier will look like this to the GPT

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

## Train and Validation Sets

In [8]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
## The larger the block size, the more context the model has to make predictions, but it also increases the computational complexity and memory requirements of the model.
## Block size denote the number of characters we want to look at in the past to predict the next character. It can also be called context limit.
block_size = 8
print(train_data[:block_size+1]) ## First 9 characters in the sequence, the first 8 characters will be the input and the last character will be the target

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [10]:
## Examples of input and target pairs
"""
We start with context of size 1 and keep increasing it until we reach the block size.
This is because we want to train the model to predict the next character given a context of varying size. 
"""
x = train_data[:block_size] # first 8 characters
y = train_data[1:block_size+1] # next 8 characters, shifted by one
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


## Pull chuncks of data from random parts of the sequence

In [11]:
torch.manual_seed(1337)
batch_size = 4 ## number of independent sequences to process in parallel
block_size = 8 ## maximum context length for predictions

def get_batch(split):
    ## generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) ## Find out the random starting indices for the batch. The starting index should leave enough room for a full block of input and target characters, hence the len(data) - block_size
    x = torch.stack([data[i:i+block_size] for i in ix]) ## stack the input sequences together to form each batch of size block_size
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) ## stack the target sequences together. It is last block_size characters of the input sequence shifted by one character to the right
    return x, y

xb, yb = get_batch('train')
print(f"inputs:\n{xb.shape}\n{xb}")
print(f"targets:\n{yb.shape}\n{yb}")

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

## Creating Bigram Language Model 

In [12]:
torch.manual_seed(1337)
n_embd = 32 ## Number of embedding dimensions. This is the size of the vector that will represent each character in the model. It is a hyperparameter.

## One of the simplest possible neural network for language modelling is bigram language model.
class BigramLanguageModel(nn.Module):
    ## Many things implemented here may not be useful for a bigram model (like positional embeddings), but we will keep them for the sake of understanding and future extensions to more complex models.
    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        ## construct token embedding table of size (vocab_size, n_embd)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.positional_embedding_table = nn.Embedding(block_size, n_embd) ## Positional embedding table of size (block_size, n_embd) to encode the position of each token in the input sequence.
        self.lm_head = nn.Linear(n_embd, vocab_size) ## Linear layer to project the embedding to the size of the vocabulary to get the logits for the next token
    
    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensors of integers
        B, T = idx.shape
        ## For each idx, logits will have a row from the embedidng table
        token_emb = self.token_embedding_table(idx) ## (B,T,C) --> (Batch, Time, Channel). In this case, batch=4, time=8, channel=65 (vocab size)
        ## Add positional embeddings
        token_emb = token_emb + pos_emb ## (B, T, C)
        pos_emb = self.positional_embedding_table(torch.arange(T, device=idx.device)) ## (T, C)
        x = token_emb + pos_emb ## (B, T, C) + (T, C) --> (B, T, C)
        logits = self.lm_head(x) ## (B,T,C) --> (Batch, Time, Vocab Size)

        if targets is None:
            loss = None
        else:
            ## Reshaping the logits as pytorch expects them to be in the form (B*T, C) and targets to be in the form (B*T) for cross_entropy loss function
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            ## This is pytorch version of negative log likelihood loss.
            ## It applies softmax to the logits and then computes the negative log likelihood loss between the predicted probabilities and the target indices.
            loss = F.cross_entropy(logits, targets) ## This measures the quality of logits wrt. the targets.
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        """
        This function takes in a starting index and generates a sequence of new tokens by repeatedly predicting the next token and appending it to the input sequence.
         - idx is the starting index (B, T) where B is the batch size and T is the current length of the input sequence.
         - max_new_tokens is the number of new tokens to generate.
        """
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            ## Get the predictions
            ## When we call self(idx), PyTorch's nn.Module.__call__ spesifically looks for a method named forward and calls it with the given arguments.
            ## So, self(idx) is equivalent to self.forward(idx).
            ## This will return the logits and loss for the given input indices.
            logits, loss = self(idx)
            ## Once we get the logits, we will focus only on the last time step
            logits = logits[:, -1, :] ## becomes (B, C)
            ## apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) ## (B, C)
            ## Get 1 sample from the multinomial distribution
            idx_next = torch.multinomial(probs, num_samples=1) ## (B, 1)
            ## Append sampled integers from the probability distribution to the running sequence extending it by one position. Shape goes from (B, T) to (B, T+1)
            idx = torch.cat((idx, idx_next), dim=1) ## (B, T+1)
        return idx
    
m = BigramLanguageModel()
logits, loss = m(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss}")

idx = torch.zeros((1,1), dtype=torch.long) ## Starting with a batch of size 1 and a sequence length of 1
print(f"\nGenerated text: {decode(m.generate(idx, max_new_tokens=100)[0].tolist())}")

Logits shape: torch.Size([32, 65])
Loss: 4.878634929656982

Generated text: 
SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


## Training the Bigram Model

In [13]:
## Create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32

for steps in range(10000):
    ## Sample a batch of data
    xb, yb = get_batch("train")

    ## Evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) ## set_to_none=True is a more efficient way to zero out the gradients, it avoids unnecessary memory operations.
    loss.backward() ## This computes the gradients of the loss with respect to the model parameters and stores them in the .grad attribute of each parameter.
    optimizer.step() ## This updates the model parameters based on the computed gradients and the learning rate specified in the optimizer.

print(f" Loss: {loss.item():.4f}")

 Loss: 2.3824


In [15]:
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


## The mathematical trick in self attention

In [16]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2 ## Batch size, Time steps, Channel (vocab size)
x = torch.randn(B, T, C)
print(x.shape)

torch.Size([4, 8, 2])


We want next characters to see only the characters before them while training (similar to Masked Attention). So,

### Instead of:

In [17]:
xbow = torch.zeros((B, T, C)) ## x bag of words. It is a tensor that will store the mean of the previous time steps for each time step.
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0) ## Take the mean of the previous time steps to get the current time step. This is a simple way but not very efficient.

### We will apply the mathematical multiplication using pytorch

#### Toy example illustrating how torch.tril and matrix multiplication can be used for a weighted aggregation.


In [18]:
## Version 1: Using nested loops for weighted aggregation
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3)) ## Lower triangular matrix of ones
a = a / torch.sum(a, 1, keepdim=True) ## Normalize the rows to get the average of the previous time steps
b = torch.randint(0, 10, (3, 2)).float() ## Random input tensor of shape (3, 2)
c = a @ b ## Matrix multiplication to get the weighted aggregation of the input tensor b using the lower triangular matrix a. The result will be a tensor of shape (3, 2) where each row is the average of the previous rows of b.

print(f"a=\n{a}\n--\nb=\n{b}\n--\nc=\n{c}")

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


Note that element at ith position in c is average of the elements till ith position in b

In [19]:
## Version 2: Using matric multiplication for weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
# print(wei)

xbow2 = wei @ x ## (T, T) @ (B, T, C) --> (B, T, T) @ (B, T, C) --> (B, T, C) ## Broadcasting
## xbow and xbow2 are identical, but xbow2 is computed using matrix multiplication which is more efficient than the nested for loops used to compute xbow.
torch.allclose(xbow, xbow2, atol=1e-6) ## xbow and xbow2 are not exactly equal due to floating point precision issues, but they are close enough for practical purposes.

# print(xbow[0])
# print(xbow2[0])

True

In [20]:
## Version 3: Using Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T)) ## Initializing to zeros to show weighted uniform aggregation
wei = wei.masked_fill(tril==0, float('-inf')) ## Make cells with 0 in tril as negative infinity
# print(wei)
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [21]:
## Version 4: Self Attention
torch.manual_seed(1337)
B, T, C = 4, 8, 32 ## Batch, Time, Channels
x = torch.randn(B, T, C)

## Let's see a single head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False) ## (B, T, head_size)
query = nn.Linear(C, head_size, bias=False) ## (B, T, head_size)
value = nn.Linear(C, head_size, bias=False) ## (B, T, head_size)
k = key(x) ## (B, T, 16)
q = query(x) ## (B, T, 16)
wei = q @ k.transpose(-2, -1) ##  Transpose last 2 dimensions ## (B, T, 16) @ (B, 16, T) --> (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril==0, float('-inf')) ## Upper triangular masking
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v ## (B, T, T) @ (B, T, head_size) --> (B, T, head_size)

print(out.shape)
print(out[0])

torch.Size([4, 8, 16])
tensor([[-0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
         -0.8873,  0.1907,  0.1762, -0.5943, -0.4812, -0.4860,  0.2862,  0.5710],
        [ 0.6764, -0.5477, -0.2478,  0.3143, -0.1280, -0.2952, -0.4296, -0.1089,
         -0.0493,  0.7268,  0.7130, -0.1164,  0.3266,  0.3431, -0.0710,  1.2716],
        [ 0.4823, -0.1069, -0.4055,  0.1770,  0.1581, -0.1697,  0.0162,  0.0215,
         -0.2490, -0.3773,  0.2787,  0.1629, -0.2895, -0.0676, -0.1416,  1.2194],
        [ 0.1971,  0.2856, -0.1303, -0.2655,  0.0668,  0.1954,  0.0281, -0.2451,
         -0.4647,  0.0693,  0.1528, -0.2032, -0.2479, -0.1621,  0.1947,  0.7678],
        [ 0.2510,  0.7346,  0.5939,  0.2516,  0.2606,  0.7582,  0.5595,  0.3539,
         -0.5934, -1.0807, -0.3111, -0.2781, -0.9054,  0.1318, -0.1382,  0.6371],
        [ 0.3428,  0.4960,  0.4725,  0.3028,  0.1844,  0.5814,  0.3824,  0.2952,
         -0.4897, -0.7705, -0.1172, -0.2541, -0.6892,  0.1979, -0.1513,  0.7666],

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other.
- Encoder attention block does not require masking and hance we can just delete the single line that does masking with `tril`, allowing all tokens to communicate. Triangular masking is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much.

# Creating Decoder-only Transformer

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import os

In [24]:
B, T, C = 4, 8, 32 ## Batch, Time, Channels
batch_size = 16
block_size = 64
max_iters = 1500
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
eval_iters = 50
n_embd = 128
n_head = 2
n_layer = 2
dropout = 0.2

torch.manual_seed(1337)

Using device: cpu


In [25]:
def download_dataset():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    path = r"C:\Users\sarve\Documents\Personal Experiments\GPT From Scratch\input.txt"

    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
        print(f"Downloaded: {path}")
    else:
        print(f"Already exists: {path}")
    
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    return text



In [26]:
text = download_dataset()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

Already exists: C:\Users\sarve\Documents\Personal Experiments\GPT From Scratch\input.txt


In [27]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [28]:
class Head(nn.Module):
    """ One head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) ## Registering the lower triangular matrix as a buffer
        self.dropout = nn.Dropout(dropout) ## Dropout layer to prevent overfitting
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x) ## (B, T, head_size)
        q = self.query(x) ## (B, T, head_size)
        v = self.value(x) ## (B, T, head_size)

        wei = q @ k.transpose(-2, -1) * C**-0.5 ## (B, T, head_size) @ (B, head_size, T) --> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T]==0, float('-inf')) ## Upper triangular masking for decoder blocks
        wei = F.softmax(wei, dim=-1) ## (B, T, T)
        out = self.dropout(wei)
        out = wei @ v ## (B, T, T) @ (B, T, head_size) --> (B, T, head_size)
        return out

In [29]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) ## Create a list of heads
        self.proj = nn.Linear(n_embd, n_embd) ## Linear layer to project the concatenated output of the heads back to the original channel size
        self.dropout = nn.Dropout(dropout) ## Dropout layer for regularization
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(self.proj(out))
        return out

In [30]:
class FeedForward(nn.Module):
    """ A simple Feed Forward Neural Network - Linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), ## Linear layer
            nn.ReLU(), ## Non-linearity
            nn.Linear(4 * n_embd, n_embd), ## Linear layer to project back to original channel size
            nn.Dropout(dropout) ## Dropout layer for regularization
        )
    
    def forward(self, x):
        return self.net(x)

In [31]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim) ## Scale parameter
        self.beta = torch.zeros(dim) ## Shift parameter

    def __call__(self, x):
        ## Calculate the forward pass of layer normalization
        xmean = x.mean(1, keepdim=True) ## Mean over the time dimension ## Batch mean
        xvar = x.var(1, keepdim=True) ## Variance over the time dimension ## Batch variance
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) ## Normalize the input for unit variance and zero mean
        self.out = self.gamma * xhat + self.beta ## Scale and shift the normalized input
        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]
    
module = LayerNorm1d(100)
x = torch.randn(32, 100) ## Batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [32]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size) ## Multi-head self attention
        self.ffwd = FeedForward(n_embd) ## Feed forward neural network
        self.ln1 = nn.LayerNorm(n_embd) ## Layer normalization for the input to self-attention
        self.ln2 = nn.LayerNorm(n_embd) ## Layer normalization for the input to feed forward network
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x)) ## Add the output of the self-attention to the input (residual connection)
        x = x + self.ffwd(self.ln2(x)) ## Add the output of the feed forward network to the input (residual connection)
        return x

In [33]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.positional_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) ## Final layer normalization
        self.lm_head = nn.Linear(n_embd, vocab_size)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_emb = self.token_embedding_table(idx) ## (B,T,C)
        pos_emb = self.positional_embedding_table(torch.arange(T, device=idx.device)) ## (T, C)
        x = token_emb + pos_emb ## (B, T, C) + (T, C) --> (B, T, C)
        x = self.blocks(x) ## Apply the transformer blocks
        x = self.ln_f(x) ## Apply final layer normalization
        logits = self.lm_head(x) ## (B,T,C) --> (Batch, Time, Vocab Size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] ## (B, block_size) Crop the context to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1) ## (B, T+1)
        return idx

In [34]:
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.420929 M parameters
step 0: train loss 4.3276, val loss 4.3227
step 500: train loss 2.4156, val loss 2.4228
step 1000: train loss 2.2295, val loss 2.2469
step 1499: train loss 2.1139, val loss 2.1597

Fisth hoser Win suthet feed you pham this soun ach isen?

DOR KIck my kive vores, Thareden'd su,
An ear man longlengay to re.

Nur EESER:
Is you so, I at shiw del youll we fe hell He wer
Intrinelf.
The do he and wingd thencer, tereathice the ste
the yous arget fiell tinds you
Lorn nown whe agere his themooth on.

Prour onould d dome cre, The hice chey yerverreyseir for.

GOUSTE:
Wen I the The ther, shes I se offiry ler heatshon shingto langurs,
Mecochad wors, ham fort; to lerat en,
Teate, Voreicest land enfeast shered suand; to bat no tin,
Bur?

DULOLENTING RESYBWIINCHAVING Curd?

OF:
ENBESHe:
Anntt CE sh:
Vo chor:
An lay, whe, I menceaid xolk agald all now thenokinttiens dords?
Ar yout brear,.

Bexvithis CALGADYeptas:
Wheetsho lardel. ETE bellf andood heireythy word;
Sit! hir ay mace b